In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Mobile Money Fraud Detection: Data Preparation, Evaluation and Explanations

Dataset: MoMTSim (simulated mobile money transactions), TRANSFER transactions from accounts with at least 5 earlier transfers, steps 24 and later.

In [ ]:
# Imports used by the path check
import os

# Walk the Kaggle input directory and print every CSV found, with its size in MB
for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        if name.endswith(".csv"):
            full = os.path.join(root, name)                       # full path to the file
            print(full, "|", round(os.path.getsize(full) / 1e6, 1), "MB")

## 1. Data preparation
Loads the raw file, cleans it, builds the account-history and recipient features, creates the temporal and account-disjoint splits, and trains the strict-tier models.

In [ ]:
# Imports used by the rebuild
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

# Paths: the raw file is read-only input, results are written to the working directory
RAW = "/kaggle/input/datasets/linzkibet/frauddetectiodata/Synthetic Mobile Money Transaction Dataset/MoMTSim_20240722202413_1000_dataset.csv"
OUT = "/kaggle/working"

# Settings
MIN_HISTORY = 5            # minimum earlier transfers for a row to enter the modelling scope
TRAIN_END, VAL_END = 115, 153   # temporal split cut-offs (steps)
Z_FLOOR = 0.1              # floor added to the standard deviation in the z-score
WINDOWS = [6, 24]          # windows (steps) for the recipient counts

# 1. Load the raw file and keep TRANSFER rows, the only type that contains fraud
df = pd.read_csv(RAW)
tr = df[df["transactionType"] == "TRANSFER"].copy()
tr = tr.rename_axis("row_id").reset_index()          # row_id = position in the raw file
del df                                               # free memory
tr["initiator"] = tr["initiator"].astype(str)        # identifiers stored as strings
tr["recipient"] = tr["recipient"].astype(str)

# 2. Cleaning: drop exact duplicates and non-positive amounts (outliers are kept)
dup_mask = tr.drop(columns="row_id").duplicated()
tr = tr[~dup_mask & (tr["amount"] > 0)].reset_index(drop=True)
print("Rows after cleaning:", len(tr))               # expect 2,510,889

# 3. Transformations
tr["amount_log"] = np.log1p(tr["amount"])            # log-scaled amount
tr["amount_log_sq"] = tr["amount_log"] ** 2          # helper column for the running variance
tr["hour"] = tr["step"] % 24                         # hour of day, assuming one step is one hour

# Sort by initiator then time, so every running statistic sees only earlier transfers
tr = tr.sort_values(["initiator", "step", "row_id"], kind="stable").reset_index(drop=True)
g = tr.groupby("initiator")                          # one group per initiating account

# 4. Account-history features, computed from earlier transfers only
tr["prior_count"] = g.cumcount()                                   # earlier transfers by this account
n = tr["prior_count"].where(tr["prior_count"] > 0)                 # same count, NaN when there is no history
s1 = g["amount_log"].cumsum() - tr["amount_log"]                   # sum of earlier log amounts
s2 = g["amount_log_sq"].cumsum() - tr["amount_log_sq"]             # sum of earlier squared log amounts
tr["prior_mean_log"] = s1 / n                                      # account's typical amount so far
var = (s2 / n) - tr["prior_mean_log"] ** 2                         # running variance
tr["prior_std_log"] = np.sqrt(var.clip(lower=0))                   # clip guards against tiny negative values
tr["amt_logratio_own"] = tr["amount_log"] - tr["prior_mean_log"]   # helper for the z-score
tr["amt_z_own"] = tr["amt_logratio_own"] / (tr["prior_std_log"] + Z_FLOOR)   # deviation from the account's norm
tr["steps_since_last"] = g["step"].diff()                          # steps since the account's previous transfer
tr["is_new_recipient"] = (~tr.duplicated(["initiator", "recipient"])).astype("int8")   # first send to this recipient

# 5. Balance-based features (the artefact tier)
for col in ["oldBalInitiator", "newBalInitiator", "oldBalRecipient", "newBalRecipient"]:
    tr[col + "_log"] = np.log1p(tr[col].clip(lower=0))             # log-scaled balances, negatives clipped to 0
tr["amount_to_balance"] = (tr["amount"] / tr["oldBalInitiator"].where(tr["oldBalInitiator"] > 0)).fillna(-1)

def slog(x):
    return np.sign(x) * np.log1p(np.abs(x))                        # signed log transform

tr["init_bal_diff"] = slog(tr["oldBalInitiator"] - tr["amount"] - tr["newBalInitiator"])    # initiator balance error
tr["recip_bal_diff"] = slog(tr["newBalRecipient"] - tr["oldBalRecipient"] - tr["amount"])   # recipient balance error

# 6. Windowed recipient counts: transfers received in the previous 6 and 24 steps (current step excluded)
n_steps = int(tr["step"].max()) + 1                                # number of step values (0 to max)
cnt = tr.groupby(["recipient", "step"]).size().unstack(fill_value=0)   # recipients x steps matrix of counts
cnt = cnt.reindex(columns=range(n_steps), fill_value=0)            # make sure every step has a column
cs = np.concatenate([np.zeros((len(cnt), 1)), cnt.to_numpy().cumsum(axis=1)], axis=1)   # cs[:, k] = received before step k
rec_idx = pd.Index(cnt.index).get_indexer(tr["recipient"])         # matrix row of each transfer's recipient
st = tr["step"].to_numpy()                                         # step of each transfer
for w in WINDOWS:
    lo = np.maximum(st - w, 0)                                     # window start
    tr[f"recip_cnt_{w}"] = cs[rec_idx, st] - cs[rec_idx, lo]       # transfers received in steps [st - w, st - 1]
del cnt, cs                                                        # free memory

# 7. Modelling scope, evaluation-only flags and account identifier
model_df = tr[tr["prior_count"] >= MIN_HISTORY].copy()             # history-rich transfers only
model_df["stratum_legit_unusual"] = ((model_df["isFraud"] == 0) & (model_df["amt_z_own"] > 3)).astype("int8")
model_df["acct_id"] = pd.factorize(model_df["initiator"])[0]       # pseudonymised integer account code
has_legit = (model_df["isFraud"] == 0).astype("int8").groupby(model_df["acct_id"]).transform("max")   # account has a legitimate transfer
has_fraud = (model_df["isFraud"] == 1).astype("int8").groupby(model_df["acct_id"]).transform("max")   # account has a fraud transfer
model_df["acct_mixed"] = ((has_legit == 1) & (has_fraud == 1)).astype("int8")   # account has both (evaluation only)

# 8. Temporal split by step
model_df["split_time"] = np.select(
    [model_df["step"] <= TRAIN_END, model_df["step"] <= VAL_END],
    ["train", "val"], default="test")

# 9. Account-disjoint split: 60/20/20 of accounts, shuffled separately within each tier
rng = np.random.default_rng(0)                                     # fixed seed, same as before
acct_tbl = model_df.groupby("acct_id")["acct_mixed"].first().reset_index()   # one row per account
split_map = {}                                                     # account id -> split name
for flag, grp in acct_tbl.groupby("acct_mixed"):
    ids = grp["acct_id"].to_numpy().copy()                         # accounts in this tier
    rng.shuffle(ids)                                               # random order
    n_acc = len(ids)
    n_tr, n_va = int(0.6 * n_acc), int(0.2 * n_acc)                # train and validation counts
    labels = np.array(["train"] * n_tr + ["val"] * n_va + ["test"] * (n_acc - n_tr - n_va))
    split_map.update(dict(zip(ids, labels)))                       # record the assignment
model_df["split_acct"] = model_df["acct_id"].map(split_map)        # attach the split to every row

# 10. Compare the rebuilt data with the counts printed earlier (rows: legitimate, fraud)
expected_time = {"train": (118763, 334697), "val": (62793, 115184), "test": (92886, 123387)}
expected_acct = {"train": (171086, 345428), "val": (47207, 118637), "test": (56149, 109203)}
for col, expected in [("split_time", expected_time), ("split_acct", expected_acct)]:
    ct = pd.crosstab(model_df[col], model_df["isFraud"])           # class counts per split
    got = {s: (int(ct.loc[s, 0]), int(ct.loc[s, 1])) for s in ct.index}
    print(col, "matches earlier run:", got == expected)
    print(ct)
assert len(model_df) == 847710                                     # modelling scope size from the earlier run

# 11. Tier definitions and final validation checks
FEATS_STRICT = ["amt_z_own", "hour", "steps_since_last", "is_new_recipient", "recip_cnt_6", "recip_cnt_24"]
FEATS_PROFILE = ["prior_mean_log", "prior_std_log", "prior_count"]
FEATS_BAL = ["oldBalInitiator_log", "newBalInitiator_log", "amount_to_balance", "init_bal_diff",
             "oldBalRecipient_log", "newBalRecipient_log", "recip_bal_diff"]
tiers_final = {
    "strict_behavioural": FEATS_STRICT,
    "behavioural_plus_amount": FEATS_STRICT + ["amount_log"],
    "profile": FEATS_PROFILE,
    "balance": FEATS_BAL,
}
feature_cols = sorted({f for fs in tiers_final.values() for f in fs})        # every input column, no duplicates
meta_cols = ["row_id", "step", "acct_id", "isFraud", "stratum_legit_unusual",
             "acct_mixed", "split_time", "split_acct"]                        # identification and evaluation columns
assert model_df[feature_cols].notna().all().all()                            # no missing values
assert np.isfinite(model_df[feature_cols].to_numpy()).all()                  # no infinite values
assert model_df["row_id"].is_unique                                          # every row appears once
a = model_df.groupby("split_acct")["acct_id"].agg(set)                       # accounts per split
assert not (a["train"] & a["val"]) and not (a["train"] & a["test"]) and not (a["val"] & a["test"])

# 12. Save the dataset and the tier definitions
model_df[meta_cols + feature_cols].to_parquet(f"{OUT}/model_data.parquet", index=False)
with open(f"{OUT}/feature_sets.json", "w") as fh:
    json.dump(tiers_final, fh, indent=2)

# 13. Retrain and save the strict-tier model for each split (replaced by the final evaluation cell)
data = model_df[meta_cols + feature_cols]                                    # the saved columns only
for split_col in ["split_time", "split_acct"]:
    tr_ = data[data[split_col] == "train"]
    va_ = data[data[split_col] == "val"]
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=0, verbose=-1)                       # same settings as the earlier runs
    m.fit(tr_[FEATS_STRICT], tr_["isFraud"],
          eval_set=[(va_[FEATS_STRICT], va_["isFraud"])], eval_metric="auc",
          callbacks=[lgb.early_stopping(30, verbose=False)])                 # early stopping on the validation split
    joblib.dump(m, f"{OUT}/model_strict_{split_col}.joblib")                 # saved for the explanation layer
print("Saved model_data.parquet, feature_sets.json and both strict-tier models to", OUT)

# Free the large intermediate tables before the next cells
del tr, model_df, data

## 2. Final evaluation
Common scope: steps 24 and later, so every recipient window is complete. Compares two rule baselines, an Isolation Forest and four supervised feature tiers on a temporal split and an account-disjoint split. Thresholds are set on validation legitimate rows at 1% and 5% false-positive rates. Intervals come from resampling test accounts.

In [ ]:
# Imports used by this cell
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

OUT = "/kaggle/working"
MIN_STEP = 24            # first step whose 24-step recipient window is complete
PREV = 0.01              # realistic fraud prevalence for precision and alert volume
FPRS = [0.01, 0.05]      # target false-positive rates for threshold setting
N_BOOT = 200             # account-level bootstrap resamples (lower to 100 if the cell is slow)

# Load the prepared data and keep one common scope for every method
data = pd.read_parquet(f"{OUT}/model_data.parquet")
data = data[data["step"] >= MIN_STEP].reset_index(drop=True)
print("Rows in final scope:", len(data))

# Final tier definitions; profile and balance tiers are carried over from the saved file
old = json.load(open(f"{OUT}/feature_sets.json"))
STRICT = ["amt_z_own", "hour", "steps_since_last", "is_new_recipient", "recip_cnt_24"]
tiers = {
    "strict_behavioural": STRICT,
    "behavioural_plus_amount": STRICT + ["amount_log"],
    "profile": old["profile"],
    "balance": old["balance"],
}
with open(f"{OUT}/feature_sets.json", "w") as fh:
    json.dump(tiers, fh, indent=2)                            # the explanation cell reads this file

def fit_lgb(tr_, va_, feats):
    # Gradient-boosted classifier with early stopping on the validation split
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=0, verbose=-1)
    m.fit(tr_[feats], tr_["isFraud"], eval_set=[(va_[feats], va_["isFraud"])],
          eval_metric="auc", callbacks=[lgb.early_stopping(30, verbose=False)])
    return m

def metrics(flag, y, mixed, unusual):
    # Metrics for one set of alert flags on one set of test rows
    rec = flag[y == 1].mean()                                 # recall over all fraud
    fpr = flag[y == 0].mean()                                 # false-positive rate over legitimate rows
    alerts = PREV * rec + (1 - PREV) * fpr                    # alerts per transaction at prevalence PREV
    return {
        "recall_all": rec,
        "recall_fraud_acct_fraud": flag[(y == 1) & ~mixed].mean(),   # fraud in fraud-dedicated accounts
        "recall_mixed_acct_fraud": flag[(y == 1) & mixed].mean(),    # fraud in ordinary accounts
        "fpr": fpr,
        "flag_rate_unusual_legit": flag[unusual].mean() if unusual.any() else np.nan,
        "precision_at_1pct": PREV * rec / alerts if alerts > 0 else np.nan,
        "alerts_per_1000": 1000 * alerts,
    }

def evaluate(v_s, t_s, va, te, fpr_target):
    # Threshold set on validation legitimate rows, then applied to the test scores
    thr = np.quantile(v_s[va["isFraud"].to_numpy() == 0], 1 - fpr_target)
    flag = t_s > thr                                          # alerts raised on the test split
    y = te["isFraud"].to_numpy()
    mixed = te["acct_mixed"].to_numpy() == 1
    unusual = te["stratum_legit_unusual"].to_numpy() == 1
    out = metrics(flag, y, mixed, unusual)

    # Account-level bootstrap: resample test accounts with replacement, recompute recall and FPR
    acc = te["acct_id"].to_numpy()
    idx_by_acct = pd.Series(np.arange(len(te))).groupby(acc).apply(np.asarray)   # row positions per account
    rng = np.random.default_rng(0)
    boots = []
    for _ in range(N_BOOT):
        pick = rng.choice(len(idx_by_acct), size=len(idx_by_acct), replace=True)  # resample accounts
        rows_ = np.concatenate(idx_by_acct.iloc[pick].to_numpy())                 # all rows of the picked accounts
        b = metrics(flag[rows_], y[rows_], mixed[rows_], unusual[rows_])
        boots.append((b["recall_all"], b["fpr"]))
    boots = np.array(boots)
    out["recall_all_lo"], out["recall_all_hi"] = np.nanpercentile(boots[:, 0], [2.5, 97.5])
    out["fpr_lo"], out["fpr_hi"] = np.nanpercentile(boots[:, 1], [2.5, 97.5])
    return out

rows = []
for split_col in ["split_time", "split_acct"]:
    # Training, validation and test rows for this split definition
    tr_ = data[data[split_col] == "train"]
    va_ = data[data[split_col] == "val"]
    te_ = data[data[split_col] == "test"]
    print(split_col, "test rows:", len(te_))

    scores = {}   # method name -> (validation scores, test scores); higher means more suspicious

    # Baselines: fixed amount rule, own-norm deviation rule, Isolation Forest on the strict tier
    scores["rule: amount"] = (va_["amount_log"].to_numpy(), te_["amount_log"].to_numpy())
    scores["rule: own-norm"] = (va_["amt_z_own"].abs().to_numpy(), te_["amt_z_own"].abs().to_numpy())
    iso = IsolationForest(n_estimators=200, max_samples=50_000, random_state=0, n_jobs=-1)
    iso.fit(tr_.loc[tr_["isFraud"] == 0, STRICT])             # fitted on legitimate rows only
    scores["isolation forest"] = (-iso.score_samples(va_[STRICT]), -iso.score_samples(te_[STRICT]))

    # Supervised models, one per final tier; the strict-tier model is saved for the explanation layer
    for name, feats in tiers.items():
        m = fit_lgb(tr_, va_, feats)
        scores[f"lgbm: {name}"] = (m.predict_proba(va_[feats])[:, 1], m.predict_proba(te_[feats])[:, 1])
        if name == "strict_behavioural":
            joblib.dump(m, f"{OUT}/model_strict_{split_col}.joblib")

    # Evaluate every method at every target false-positive rate
    for method, (v_s, t_s) in scores.items():
        auc = roc_auc_score(te_["isFraud"], t_s)              # threshold-free ranking quality
        for fpr in FPRS:
            r = evaluate(v_s, t_s, va_, te_, fpr)
            r.update({"split": split_col, "method": method, "fpr_target": fpr, "roc_auc": auc})
            rows.append(r)

# Collect, print and save the results table
cols = ["split", "method", "fpr_target", "roc_auc", "fpr", "fpr_lo", "fpr_hi",
        "recall_all", "recall_all_lo", "recall_all_hi",
        "recall_fraud_acct_fraud", "recall_mixed_acct_fraud",
        "flag_rate_unusual_legit", "alerts_per_1000", "precision_at_1pct"]
res = pd.DataFrame(rows)[cols]
res.to_csv(f"{OUT}/final_results.csv", index=False)
print(res.round(3).to_string(index=False))

## 3. Explanations for the strict behavioural model
SHAP contributions for the account-disjoint model, direction tables, and alert text built from the contributions. A factor is cited only if it raises the score and its value lies in the outer 5% tail of legitimate training transfers.

In [ ]:
# Imports used by this cell
import json
import joblib
import numpy as np
import pandas as pd
import shap

OUT = "/kaggle/working"
SPLIT = "split_acct"      # split whose strict-tier model is explained ("split_time" is also available)

# Load the prepared data, the tier definitions and the saved strict-tier model
data = pd.read_parquet(f"{OUT}/model_data.parquet")

# Keep the same scope as the final evaluation (complete recipient windows)
data = data[data["step"] >= 24].reset_index(drop=True)

# Confirm the scope: this should print 788922, the same count as in the final evaluation cell
print("Rows in explanation scope:", len(data))

tiers = json.load(open(f"{OUT}/feature_sets.json"))
FEATS = tiers["strict_behavioural"]
model = joblib.load(f"{OUT}/model_strict_{SPLIT}.joblib")

# Train, validation and test rows for the chosen split
tr_ = data[data[SPLIT] == "train"]
va_ = data[data[SPLIT] == "val"]
te_ = data[data[SPLIT] == "test"]

# 1. SHAP values on a random sample of test rows
sample = te_.sample(n=min(20000, len(te_)), random_state=0).reset_index(drop=True)
explainer = shap.TreeExplainer(model)                          # exact SHAP values for tree models
sv = explainer.shap_values(sample[FEATS])
if isinstance(sv, list):                                       # some shap versions return one array per class
    sv = sv[1]
elif sv.ndim == 3:                                             # others return a 3-D array
    sv = sv[:, :, 1]

# Fidelity check: base value plus contributions must reproduce the model's raw log-odds
ev = np.atleast_1d(explainer.expected_value)[-1]
raw = model.predict(sample[FEATS], raw_score=True)
print("Max SHAP reconstruction error:", float(np.abs(ev + sv.sum(axis=1) - raw).max()))

# Mean absolute SHAP value per feature, ranked
imp = pd.Series(np.abs(sv).mean(axis=0), index=FEATS).sort_values(ascending=False)
print("Mean |SHAP| per feature:\n", imp.round(4), "\n")

# 2. Direction tables: fraud rate and mean SHAP value across bins of each feature
shap_df = pd.DataFrame(sv, columns=FEATS)                      # SHAP values aligned with the sampled rows
for f in FEATS:
    if sample[f].nunique() <= 2:
        bins = sample[f]                                       # binary feature: group by its two values
    else:
        bins = pd.qcut(sample[f], 8, duplicates="drop")        # otherwise up to eight quantile bins
    tab = (pd.DataFrame({"fraud": sample["isFraud"], "shap": shap_df[f]})
             .groupby(bins, observed=True)
             .agg(n=("fraud", "size"), fraud_rate=("fraud", "mean"), mean_shap=("shap", "mean")))
    print(f)
    print(tab.round(3))
    print()

# 3. Reference distributions: legitimate training transfers, used to place a value among its peers
legit_ref = tr_[tr_["isFraud"] == 0]
ref = {f: np.sort(legit_ref[f].to_numpy()) for f in FEATS}     # sorted values for fast percentile lookups
TAIL = 0.05                                                    # a value is "extreme" if at most 5% of legitimate rows are as extreme
CITABLE = [f for f in FEATS if f != "hour"]                    # hour is never cited (no signal in this data)

def tail_info(f, v):
    # Returns which tail the value sits in and the share of legitimate rows at least as extreme
    arr = ref[f]
    n = len(arr)
    upper = 1 - np.searchsorted(arr, v, side="left") / n       # share of legitimate rows with value >= v
    lower = np.searchsorted(arr, v, side="right") / n          # share of legitimate rows with value <= v
    return ("high", upper) if upper <= lower else ("low", lower)

def fmt(f, v, side, share):
    # Wording states the value and its position among legitimate transfers, with no claim about risk direction
    pos = f"{'above' if side == 'high' else 'below'} {100 * (1 - share):.0f}% of legitimate transfers"
    if f == "amt_z_own":
        return f"amount deviation from this account's usual transfer size (z-score {v:+.1f}, {pos})"
    if f == "steps_since_last":
        return f"{v:.0f} steps since this account's previous transfer ({pos})"
    if f == "is_new_recipient":
        return ("first transfer from this account to this recipient" if v == 1
                else "this account had sent to this recipient before")
    return f"recipient received {int(v)} transfers in the previous 24 steps ({pos})"   # recip_cnt_24

def factors(i, min_contrib=0.1, top_k=3):
    # Citable factors for row i: positive SHAP contribution and a value in the outer tail
    out = []
    for f in CITABLE:
        c = sv[i, FEATS.index(f)]                              # contribution of this feature to row i (log-odds)
        if c <= min_contrib:
            continue
        side, share = tail_info(f, sample.at[i, f])
        if share > TAIL:
            continue
        out.append((c, f, side, share))
    return sorted(out, reverse=True)[:top_k]                   # largest contributions first

scores = model.predict_proba(sample[FEATS])[:, 1]              # fraud score of each sampled row

def make_alert(i):
    fs = factors(i)
    if not fs:
        return (f"Risk score {scores[i]:.2f}. No single factor lies outside the usual range of "
                f"legitimate transfers; the score reflects a combination of factors.")
    parts = [f"{fmt(f, sample.at[i, f], side, share)} (+{c:.2f})" for c, f, side, share in fs]
    return f"Risk score {scores[i]:.2f}. Factors: " + "; ".join(parts) + "."

# 4. Alerts at a 1% false-positive threshold set on validation legitimate rows
v_s = model.predict_proba(va_[FEATS])[:, 1]                    # validation scores
thr = np.quantile(v_s[va_["isFraud"].to_numpy() == 0], 0.99)   # score cut-off for alerts
flagged = np.where(scores > thr)[0]                            # sampled rows that raise an alert
print("Alerts raised in the sample:", len(flagged), "of", len(sample), "\n")

def show(i):
    # The true label and account tier are printed for review only and never appear in the alert text
    print(f"[review only: true label {int(sample.at[i, 'isFraud'])}, "
          f"mixed account {int(sample.at[i, 'acct_mixed'])}]", make_alert(i))

top5 = list(np.argsort(scores)[::-1][:5])                                     # five highest-scoring rows
false5 = [i for i in flagged if sample.at[i, "isFraud"] == 0][:5]             # first five false alarms
rand5 = list(np.random.default_rng(1).choice(flagged, size=min(5, len(flagged)), replace=False))
for i in top5 + false5 + rand5:
    show(i)

# 5. Do true and false alerts cite the same factors? (the key limit for the report)
fl = [factors(i) for i in flagged]                             # citable factors of every alert
summ = pd.DataFrame({
    "label": sample.loc[flagged, "isFraud"].to_numpy(),        # 1 = fraud, 0 = legitimate
    "mixed": sample.loc[flagged, "acct_mixed"].to_numpy(),     # 1 = ordinary account
    "has_factor": [len(x) > 0 for x in fl],                    # at least one citable factor
    "first": [x[0][1] if x else "none" for x in fl],           # largest citable factor
})
print("\nAlerts by label and account tier; share with at least one citable factor:")
print(summ.groupby(["label", "mixed"]).agg(n=("has_factor", "size"), share_with_factor=("has_factor", "mean")).round(3))
print("\nLargest cited factor by label:")
print(pd.crosstab(summ["first"], summ["label"]))

# Save the example alert texts for the report appendix (true label included for review only)
ex = pd.DataFrame({
    "row_id": sample.loc[top5 + false5 + rand5, "row_id"].to_numpy(),
    "true_label": sample.loc[top5 + false5 + rand5, "isFraud"].to_numpy(),
    "alert_text": [make_alert(i) for i in top5 + false5 + rand5],
})
ex.to_csv(f"{OUT}/example_alerts.csv", index=False)

### Alert wording
Replaces the percentile wording, handles singular forms, and adds the repeat-recipient factor that lowers the score. Reuses the variables from the previous cell.

In [ ]:
# Replacement wording helpers; this cell reuses sample, sv, scores, FEATS, factors,
# top5, false5, rand5, flagged and OUT from the explanation cell
import numpy as np
import pandas as pd

def plural(n, word):
    # "1 transfer" versus "3 transfers"
    return f"{n} {word}" if n == 1 else f"{n} {word}s"

def pos_text(side, share):
    # Position of a value among legitimate reference transfers, avoiding the misleading "100%" rounding
    if share < 0.001:                                    # fewer than 0.1% of reference transfers are this extreme
        return "more extreme than any legitimate transfer in the reference data"
    pct = 100 * (1 - share)                              # share of legitimate transfers the value exceeds
    pct_txt = f"{pct:.1f}%" if share < 0.01 else f"{pct:.0f}%"   # one decimal in the extreme tail
    return f"{'higher' if side == 'high' else 'lower'} than {pct_txt} of legitimate transfers"

def fmt(f, v, side, share):
    # States the value and its position among legitimate transfers, with no claim about risk direction
    pos = pos_text(side, share)
    if f == "amt_z_own":
        return f"amount deviation from this account's usual transfer size (z-score {v:+.1f}, {pos})"
    if f == "steps_since_last":
        return f"{plural(int(v), 'step')} since this account's previous transfer ({pos})"
    if f == "is_new_recipient":                          # safeguard: this feature is rarely in the outer tail
        return ("first transfer from this account to this recipient" if v == 1
                else "this account had sent to this recipient before")
    return f"recipient received {plural(int(v), 'transfer')} in the previous 24 steps ({pos})"   # recip_cnt_24

def make_alert(i):
    # Factors raising the score (outer-tail values only), plus the repeat-recipient factor that lowers it
    fs = factors(i)                                      # citable factors defined in the explanation cell
    if fs:
        parts = [f"{fmt(f, sample.at[i, f], side, share)} (+{c:.2f})" for c, f, side, share in fs]
        text = f"Risk score {scores[i]:.2f}. Factors raising the score: " + "; ".join(parts) + "."
    else:
        text = (f"Risk score {scores[i]:.2f}. No single factor lies outside the usual range of "
                f"legitimate transfers; the score reflects a combination of factors.")
    c_rep = sv[i, FEATS.index("is_new_recipient")]       # contribution of the recipient-novelty feature
    if sample.at[i, "is_new_recipient"] == 0 and c_rep < -0.3:
        text += f" Factor lowering the score: this account had sent to this recipient before ({c_rep:+.2f})."
    return text

# Reprint the same 15 example alerts with the corrected wording (label shown for review only)
for i in top5 + false5 + rand5:
    print(f"[review only: true label {int(sample.at[i, 'isFraud'])}, "
          f"mixed account {int(sample.at[i, 'acct_mixed'])}]", make_alert(i))

# Re-save the appendix examples
sel = top5 + false5 + rand5
pd.DataFrame({
    "row_id": sample.loc[sel, "row_id"].to_numpy(),
    "true_label": sample.loc[sel, "isFraud"].to_numpy(),
    "alert_text": [make_alert(i) for i in sel],
}).to_csv(f"{OUT}/example_alerts.csv", index=False)

# How often does the repeat-recipient factor appear in true and false alerts?
rep = ((sample.loc[flagged, "is_new_recipient"] == 0).to_numpy()
       & (sv[flagged, FEATS.index("is_new_recipient")] < -0.3))
print(pd.crosstab(sample.loc[flagged, "isFraud"].to_numpy(), rep,
                  rownames=["true label"], colnames=["repeat-recipient factor shown"]))

## 4. Scoring bundle for the dashboard
Saves the operating thresholds, the reference distributions used for alert wording, and per-row scores, contributions and alert text for the account-disjoint strict model.

In [ ]:
# Imports used by this cell
import json
import joblib
import numpy as np
import pandas as pd
import shap

OUT = "/kaggle/working"
SPLIT = "split_acct"       # account-disjoint strict-tier model
TAIL = 0.05                # a value is "extreme" if at most 5% of legitimate rows are as extreme
N_UNFLAGGED = 5000         # random unflagged test rows kept for comparison in the dashboard

# Load the data (same scope as the final evaluation), the tier definitions and the saved model
data = pd.read_parquet(f"{OUT}/model_data.parquet")
data = data[data["step"] >= 24].reset_index(drop=True)
tiers = json.load(open(f"{OUT}/feature_sets.json"))
FEATS = tiers["strict_behavioural"]
model = joblib.load(f"{OUT}/model_strict_{SPLIT}.joblib")
tr_ = data[data[SPLIT] == "train"]
va_ = data[data[SPLIT] == "val"]
te_ = data[data[SPLIT] == "test"].reset_index(drop=True)

# 1. Operating thresholds: score cut-offs that flag 1% and 5% of validation legitimate rows
v_s = model.predict_proba(va_[FEATS])[:, 1]
legit_val = v_s[va_["isFraud"].to_numpy() == 0]
thresholds = {"fpr_1pct": float(np.quantile(legit_val, 0.99)),
              "fpr_5pct": float(np.quantile(legit_val, 0.95))}
THR = thresholds["fpr_1pct"]                                    # threshold used for alerts below
print("Thresholds:", thresholds)

# 2. Reference distributions: legitimate training transfers, used to place a value among its peers
legit_ref = tr_[tr_["isFraud"] == 0]
CITABLE = ["amt_z_own", "steps_since_last", "recip_cnt_24"]     # features cited in alert text
ref = {f: np.sort(legit_ref[f].to_numpy()) for f in CITABLE}    # sorted values for exact percentile lookups

def tail_info(f, v):
    # Which tail the value sits in, and the share of legitimate rows at least as extreme
    arr = ref[f]
    upper = 1 - np.searchsorted(arr, v, side="left") / len(arr)
    lower = np.searchsorted(arr, v, side="right") / len(arr)
    return ("high", upper) if upper <= lower else ("low", lower)

def plural(n, word):
    # "1 transfer" versus "3 transfers"
    return f"{n} {word}" if n == 1 else f"{n} {word}s"

def pos_text(side, share):
    # Position of a value among legitimate reference transfers
    if share < 0.001:
        return "more extreme than any legitimate transfer in the reference data"
    pct = 100 * (1 - share)
    pct_txt = f"{pct:.1f}%" if share < 0.01 else f"{pct:.0f}%"
    return f"{'higher' if side == 'high' else 'lower'} than {pct_txt} of legitimate transfers"

def fmt(f, v, side, share):
    # States the value and its position among legitimate transfers, with no claim about risk direction
    pos = pos_text(side, share)
    if f == "amt_z_own":
        return f"amount deviation from this account's usual transfer size (z-score {v:+.1f}, {pos})"
    if f == "steps_since_last":
        return f"{plural(int(v), 'step')} since this account's previous transfer ({pos})"
    return f"recipient received {plural(int(v), 'transfer')} in the previous 24 steps ({pos})"   # recip_cnt_24

# 3. Score every test row and choose rows for the dashboard: all alerts plus random unflagged rows
scores = model.predict_proba(te_[FEATS])[:, 1]
flag_idx = np.where(scores > THR)[0]                            # rows that raise an alert
unflag_idx = np.where(scores <= THR)[0]                         # rows that do not
pick = np.random.default_rng(0).choice(unflag_idx, size=min(N_UNFLAGGED, len(unflag_idx)), replace=False)
sel = np.sort(np.concatenate([flag_idx, pick]))
sub = te_.iloc[sel].reset_index(drop=True)
sub_scores = scores[sel]
print("Test rows:", len(te_), "| alerts:", len(flag_idx), "| rows kept:", len(sub))

# 4. SHAP contributions for the kept rows
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(sub[FEATS])
if isinstance(sv, list):                                        # some shap versions return one array per class
    sv = sv[1]
elif sv.ndim == 3:                                              # others return a 3-D array
    sv = sv[:, :, 1]
raw = model.predict(sub[FEATS], raw_score=True)
ev = np.atleast_1d(explainer.expected_value)[-1]
print("Max SHAP reconstruction error:", float(np.abs(ev + sv.sum(axis=1) - raw).max()))

# 5. Alert text: cite outer-tail factors that raise the score, plus the repeat-recipient factor that lowers it
def alert_text(i):
    cites = []
    for f in CITABLE:
        c = sv[i, FEATS.index(f)]                               # contribution of this feature (log-odds)
        if c <= 0.1:
            continue
        side, share = tail_info(f, sub.at[i, f])
        if share <= TAIL:
            cites.append((c, f, side, share))
    cites = sorted(cites, reverse=True)[:3]                     # largest contributions first
    if cites:
        parts = [f"{fmt(f, sub.at[i, f], side, share)} (+{c:.2f})" for c, f, side, share in cites]
        text = f"Risk score {sub_scores[i]:.2f}. Factors raising the score: " + "; ".join(parts) + "."
    else:
        text = (f"Risk score {sub_scores[i]:.2f}. No single factor lies outside the usual range of "
                f"legitimate transfers; the score reflects a combination of factors.")
    c_rep = sv[i, FEATS.index("is_new_recipient")]
    if sub.at[i, "is_new_recipient"] == 0 and c_rep < -0.3:
        text += f" Factor lowering the score: this account had sent to this recipient before ({c_rep:+.2f})."
    return text

is_alert = sub_scores > THR
texts = [alert_text(i) if is_alert[i] else "" for i in range(len(sub))]   # text only for alerts

# 6. Dashboard table: identifiers are pseudonymised, labels are for evaluation only
out = pd.DataFrame({
    "row_id": sub["row_id"], "step": sub["step"], "acct_id": sub["acct_id"],
    "risk_score": sub_scores, "alert": is_alert.astype(int), "alert_text": texts,
})
for f in FEATS:
    out[f] = sub[f].to_numpy()                                  # feature values shown beside each alert
    out[f"shap_{f}"] = sv[:, FEATS.index(f)]                    # contribution of each feature to the score
out["eval_only_true_label"] = sub["isFraud"].to_numpy()         # never shown as part of an alert
out["eval_only_mixed_account"] = sub["acct_mixed"].to_numpy()
out.to_csv(f"{OUT}/dashboard_alerts.csv", index=False)

# 7. Bundle with everything else the dashboard needs to reproduce the scoring and the wording
bundle = {
    "model_file": f"model_strict_{SPLIT}.joblib",
    "features": FEATS,
    "citable_features": CITABLE,
    "thresholds": thresholds,
    "tail_share": TAIL,
    "reference_quantiles": {f: np.quantile(ref[f], np.linspace(0, 1, 1001)).tolist() for f in CITABLE},
    "note": ("Simulated data. Alerts indicate patterns associated with fraud-dedicated accounts in the "
             "training data, not confirmed fraud, and not deviation from an individual account's own habits."),
}
with open(f"{OUT}/scoring_bundle.json", "w") as fh:
    json.dump(bundle, fh)
print("Saved dashboard_alerts.csv and scoring_bundle.json")
print(out[out["alert"] == 1].head(3)[["row_id", "risk_score", "alert_text"]].to_string(index=False))

In [ ]:
# Imports used by this cell
import sys
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import sklearn

OUT = "/kaggle/working"

# Export each strict-tier model as a portable LightGBM text file, keeping only the best iteration
for split_col in ["split_time", "split_acct"]:
    m = joblib.load(f"{OUT}/model_strict_{split_col}.joblib")                   # sklearn-style model saved earlier
    m.booster_.save_model(f"{OUT}/model_strict_{split_col}.txt",
                          num_iteration=m.best_iteration_)                      # best iteration from early stopping
    print(split_col, "best iteration:", m.best_iteration_)

# Verify the exported file reproduces the original model's probabilities on 2,000 test rows
data = pd.read_parquet(f"{OUT}/model_data.parquet")
data = data[(data["step"] >= 24) & (data["split_acct"] == "test")]              # account-split test rows
FEATS = ["amt_z_own", "hour", "steps_since_last", "is_new_recipient", "recip_cnt_24"]   # final strict tier
rows = data.sample(n=2000, random_state=0)
m = joblib.load(f"{OUT}/model_strict_split_acct.joblib")
booster = lgb.Booster(model_file=f"{OUT}/model_strict_split_acct.txt")          # reload from the text file
diff = np.abs(m.predict_proba(rows[FEATS])[:, 1] - booster.predict(rows[FEATS])).max()
print("Max probability difference after export:", diff)                         # expect about 1e-9 or smaller

# Library versions, so the local environment can match
print("python", sys.version.split()[0], "| lightgbm", lgb.__version__, "| sklearn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)